# Snowflake Catalog Linked Database (CLD) Lab Guide

This notebook walks you through setting up a **Catalog Linked Database (CLD)** using AWS Glue Iceberg REST catalog with **Vended Vredentials** (Lake Formation credential vending).

The setup uses two IAM roles:
- **Snowflake role** — SigV4 auth for the Glue REST API
- **Lake Formation role** — S3 credential vending via Lake Formation

**Prerequisites:**
- AWS account with Glue, Lake Formation, and IAM access
- An existing Glue database with Iceberg tables
- An S3 bucket containing the Iceberg data files
- Snowflake ACCOUNTADMIN (or a role with CREATE INTEGRATION + CREATE DATABASE privileges)

> ### IMPORTANT
> Setup Glue DB with Lake Formation on your AWS account using `task bronze:all` from within [Lab Sources](https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines) repo

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these variables via Jinja templating — change them once and everything updates.

> ### 🛑 STOP — Update the variables above before proceeding!
> 
> Make sure you have set **all variables** to match your environment, then **run the cell below** before continuing. All subsequent SQL cells depend on these values.

In [ ]:
SF_ROLE = 'ACCOUNTADMIN'
CATALOG_INTEGRATION_NAME = 'ksampath_glue_rest_catalog_int'
GLUE_DB = 'ksampath_balloon_pops'
GLUE_NAMESPACE = GLUE_DB
AWS_ACCOUNT_ID = '849350360261'
SNOWFLAKE_IAM_ROLE = 'ksampath_snowflake_glue_catalog_read'
AWS_REGION = 'us-west-2'
LF_IAM_ROLE = 'ksampath-lf-data-access'
S3_BUCKET = 'ksampath-balloon-bronze'
CLD_DATABASE = 'balloon_game_events'
GLUE_TABLE = 'balloon_game_events'



## Step 2: Set Role and Create Catalog Integration

In [ ]:
%%sql -r use_role_result
USE ROLE {{SF_ROLE}};

## Step 2a: Create Catalog Integration (Disabled)

> **WARNING:** Do NOT re-run CREATE CATALOG INTEGRATION as it rotates the external ID. If you already created or provisioned as part of your account, skip to **Step 2b**.

We create the integration as **`ENABLED = FALSE`** first. This lets Snowflake generate the trust policy values (`API_AWS_IAM_USER_ARN`, `API_AWS_EXTERNAL_ID`) without trying to connect to Glue — which would fail since the IAM roles don't exist yet.

After deploying the CloudFormation stack (Step 2b), we'll enable it.

Key settings:
- **SIGV4** authentication — Snowflake signs Glue REST requests using your IAM role
- **`ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS`** — this is the critical setting

> **Why Vended Credentials?** ([docs](https://docs.snowflake.com/en/user-guide/tables-iceberg-configure-catalog-integration-vended-credentials))
>
> With vended credentials, **Lake Formation issues short-lived, scoped S3 credentials** directly to Snowflake — no external volume needed. This means:
>
> - **No storage integration or external volume to manage** — the catalog handles all S3 access
> - **Fine-grained access control** — Lake Formation controls which tables/columns Snowflake can read, using its existing permission model
> - **Credential scoping** — each credential is scoped to a specific table's S3 location and expires automatically
>
> The default is `EXTERNAL_VOLUME_CREDENTIALS`. If you omit `ACCESS_DELEGATION_MODE`, Snowflake falls back to using an external volume for S3 access.

**Try it with Cortex Code:** Click the SQL cell below, press **Cmd+K** / **Ctrl+K**, and paste this prompt:

> Create a catalog integration for AWS Glue Iceberg REST. Use the values from the Variables cell for the integration name, namespace, AWS account ID, IAM role, and region. Use SigV4 authentication, VENDED_CREDENTIALS for access delegation. Create it as DISABLED (ENABLED = FALSE) since the IAM roles don't exist yet.

In [ ]:
%%sql -r create_catalog_int_result
-- TODO: Use the Cortex Code prompt from the cell above (Cmd+K / Ctrl+K) to generate this SQL
-- It will fill in the correct values from the Variables cell

---

## Step 2b: Deploy AWS Resources via CloudFormation

Now that the integration exists (disabled), we can extract the trust policy values and deploy the AWS resources.

Use **Cortex Code chat**  with the following prompt:

> Describe the catalog integration from the Variables cell, extract the `API_AWS_IAM_USER_ARN` and `API_AWS_EXTERNAL_ID` values, then generate an `aws cloudformation deploy` command for template [`cfn-snowflake-cld.yaml`](https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines/tree/main/templates/aws/cfn-snowflake-cld.yaml), stack name `snowflake-cld-iam`, with `CAPABILITY_NAMED_IAM`. Use the values from the Variables cell for SnowflakeIAMRoleName, LFIAMRoleName, GlueDatabase, S3Bucket, AWSAccountId, and the region. Use the extracted API_AWS_IAM_USER_ARN and API_AWS_EXTERNAL_ID for SnowflakeUserArn and SnowflakeExternalId parameters.

The stack creates:
1. **Snowflake IAM role** with the correct trust policy (using the real values)
2. **Lake Formation IAM role** for vending S3 credentials
3. **Lake Formation S3 registration** with hybrid access
4. **Lake Formation permissions** (DESCRIBE on database, SELECT+DESCRIBE on all tables)

> **Wait 1-2 minutes** for the stack to complete and IAM to propagate.

---

## Step 2c: Verify Lake Formation Settings

After the CloudFormation stack completes, verify these Lake Formation settings in the AWS Console. If any are misconfigured, the CLD will fail with `Failed to retrieve credentials from the Catalog`.

**1. Data Lake Location (Lake Formation > Data lake locations):**

| Setting | Required Value | Notes |
|---|---|---|
| S3 location | `s3://<your-bucket>` | Must cover the path where Iceberg data files reside |
| IAM role | Lake Formation role (e.g. `ksampath-lf-data-access`) | Must be assumable by `lakeformation.amazonaws.com` with S3 read access |
| Permission mode | **Lake Formation** | **Not Hybrid** — most common cause of vended credentials failures |

**2. Glue Database settings:**
- "Use only IAM access control for new tables" — does **not** affect vended credentials (can be checked or unchecked)

**3. Lake Formation Data Permissions (SELECT/DESCRIBE on tables):**
- Controls which schemas/tables appear in the CLD during auto-discovery
- **Not** required for vended credentials to function

**4. Application Integration Settings:**
- Open **Lake Formation** > **Administration** > **Application integration settings**
- Confirm **"Allow external engines to access data in Amazon S3 locations with full table access"** is **enabled**
- This is mandatory for Snowflake's vended-credentials flow

**1. Data Lake Location**

![Data Lake Location](images/aws_lf_data_lake_settings.png)

**2. Database Settings**

![Database Settings](images/aws_lf_database_settings.png)

**3. Data Permissions**

![Data Permissions](images/aws_lf_data_permissions.png)

**4. App Integration Settings**

![App Integration Settings](images/aws-lf-app-int-settings.png)

---

## Step 2d: Enable Catalog Integration and Verify

Now that the IAM roles and Lake Formation settings are in place, enable the integration and verify it can connect to Glue.

In [ ]:
%%sql -r enable_int_result
ALTER CATALOG INTEGRATION {{CATALOG_INTEGRATION_NAME}} SET ENABLED = TRUE;

Verify the catalog integration can connect to Glue. Expect `"success": true` — if it fails, check the IAM role trust policy and Lake Formation permissions.

In [ ]:
%%sql -r verify_int_result
SELECT SYSTEM$VERIFY_CATALOG_INTEGRATION('{{CATALOG_INTEGRATION_NAME}}');

---

## Step 3: Grant Integration Usage and Create CLD

### 3.1 Grant Integration Usage

In [ ]:
%%sql -r grant_usage_result
GRANT USAGE ON INTEGRATION {{CATALOG_INTEGRATION_NAME}} TO ROLE {{SF_ROLE}};

### 3.2 Create Catalog-Linked Database

Let Cortex Code write this one for you! Click the empty SQL cell below, press **Cmd+K** / **Ctrl+K**, and paste this prompt:

> Create a Catalog-Linked Database using the values from the Variables cell for the database name and catalog integration name. Add a comment describing it as a Glue Iceberg CLD with vended credentials. Do not use EXTERNAL_VOLUME since we are using vended credentials.

Review the generated SQL, then **run the cell**.

> **Important:** After `CREATE OR REPLACE DATABASE`, you must re-run `GRANT USAGE ON INTEGRATION` if the CLD is ever recreated. See [BCR-2114](https://docs.snowflake.com/en/release-notes/bcr-bundles/2025_07/bcr-2114) for details.

In [ ]:
%%sql -r cld_result
-- Use Cortex Code (Cmd+K / Ctrl+K) to generate the CREATE DATABASE SQL here

---

## Step 4: Check Link Status and Resume Discovery

Check the CLD Link status to verify link us up and syncing

In [ ]:
%%sql -r link_status_result
SELECT SYSTEM$CATALOG_LINK_STATUS('{{CLD_DATABASE}}');

Check the CLD configuration to verify integration and catalog settings:

In [ ]:
%%sql -r cld_config_result
SELECT SYSTEM$GET_CATALOG_LINKED_DATABASE_CONFIG('{{CLD_DATABASE}}');

If the link status shows failures, use `RESUME DISCOVERY` to retry table/schema discovery. Note: this only retries discovery — if AWS-side settings (Lake Formation, IAM) changed, you must `CREATE OR REPLACE DATABASE` instead.

In [ ]:
%%sql -r resume_discovery_result
ALTER DATABASE {{CLD_DATABASE}} RESUME DISCOVERY;

---
## Step 5: Validate

List the schemas discovered in the linked Glue namespace. These are auto-synced from the Glue catalog.

In [ ]:
%%sql -r show_schemas_result
SHOW SCHEMAS IN DATABASE {{CLD_DATABASE}};

List the Iceberg tables discovered in the linked Glue namespace. These are auto-synced from the Glue catalog.

In [ ]:
%%sql -r show_tables_result
SHOW ICEBERG TABLES IN SCHEMA {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}";

Query the Iceberg table data directly through Snowflake. The `event` column contains JSON — we use `PARSE_JSON` to extract individual fields.

In [ ]:
%%sql -r sample_data_result
SELECT
  PARSE_JSON(event):player::STRING AS player,
  PARSE_JSON(event):balloon_color::STRING AS balloon_color,
  PARSE_JSON(event):score::INTEGER AS score,
  PARSE_JSON(event):event_ts::TIMESTAMP_TZ AS event_ts
FROM {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}"."{{GLUE_TABLE}}"
LIMIT 10;

### Try it: Ask Cortex Code a question about your data

Click the empty SQL cell below, press **Cmd+K** / **Ctrl+K**, and try one of these prompts:

> Which player has the highest score in the balloon_game_events table? Parse the JSON event column to extract player and score fields.

> Show the top 5 most popular balloon colors by total pops from the balloon_game_events table.

> What is the average score per player, and how many bonus pops did each player get? Parse the JSON event column.

Or make up your own question about the data!

---

## What Just Happened?

You now have Snowflake reading Iceberg tables **directly from AWS Glue** — no data copy, no ETL pipeline.

| What | How |
|---|---|
| **Catalog Integration** | Snowflake connects to Glue via the Iceberg REST API using SigV4 authentication |
| **Vended Credentials** | Lake Formation issues short-lived S3 credentials — no external volume needed |
| **Catalog-Linked Database** | Glue namespaces and tables auto-sync as Snowflake schemas and Iceberg tables |
| **Live Queries** | `SELECT` reads the latest committed Iceberg snapshots in real time |

The CLD stays in sync automatically. When new tables or schemas appear in Glue, Snowflake discovers them without any manual refresh.

**Next up:** Build Dynamic Iceberg Tables to transform this raw bronze JSON into production-ready silver aggregates.

---

## Cleanup

### Snowflake cleanup

In [ ]:
%%sql -r cleanup_result
USE ROLE {{SF_ROLE}};
DROP DATABASE IF EXISTS {{CLD_DATABASE}};
DROP CATALOG INTEGRATION IF EXISTS {{CATALOG_INTEGRATION_NAME}};

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"""
### AWS Cleanup

Since all AWS resources were created via CloudFormation, a single command tears everything down:

```bash
aws cloudformation delete-stack \\
  --stack-name snowflake-cld-iam \\
  --region {AWS_REGION}
```

This deletes the IAM roles, policies, Lake Formation resource registration, and Lake Formation permissions.

> **Note:** To monitor deletion progress:
> ```bash
> aws cloudformation wait stack-delete-complete \\
>   --stack-name snowflake-cld-iam \\
>   --region {AWS_REGION}
> ```

> **Troubleshooting:** If stack deletion fails (e.g., LF resource has active dependencies), check the CloudFormation **Events** tab for the failed resource. You may need to manually deregister the S3 location in Lake Formation before retrying the delete.
"""))

---

## Troubleshooting



### "Failed to retrieve credentials from the Catalog"

If the link status shows this error:
```json
{"failureDetails":[{"errorCode":"094120","errorMessage":"SQL Execution Error: Failed to retrieve credentials from the Catalog for table ..."}],"executionState":"RUNNING"}
```

**Root cause:** The Lake Formation Data Lake Location is not configured correctly for credential vending.

**Required Lake Formation setting (Data Lake Locations):**
- Permission mode must be **Lake Formation** (not Hybrid)
- The location must be registered with an IAM role that Lake Formation can assume and that has S3 read access

**Important notes:**
- `ALTER DATABASE ... RESUME DISCOVERY` does **NOT** re-establish the catalog connection. It only retries table/schema discovery.
- After changing Lake Formation settings, you **must** run `CREATE OR REPLACE DATABASE ... LINKED_CATALOG` to re-establish the link.
- After `CREATE OR REPLACE DATABASE`, you must re-run `GRANT USAGE ON INTEGRATION` (see steps above).